In [1]:
# ============================================================
# exp_20260428_061b_ashish_xgb_orig_te_bias_refined
# Purpose:
#   Apply bias refinement to Ashish Singh Rawat's shared XGB LB0.98109 notebook reproduction.
# ============================================================

In [2]:
from __future__ import annotations

import os
import gc
import json
import random
import warnings
from pathlib import Path
from itertools import combinations

import numpy as np
import pandas as pd
import torch

from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import TargetEncoder
from sklearn.metrics import balanced_accuracy_score, classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_sample_weight

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

In [3]:
# ============================================================
# Config
# ============================================================

class CFG:
    EXP_ID = "exp_20260429_061b_ashish_xgb_orig_te_bias_refined"
    EXP_NAME = "ashish_xgb_orig_te_bias_refined"

    SAVE_DIR = Path(f"/kaggle/working/{EXP_ID}")
    SAVE_DIR.mkdir(parents=True, exist_ok=True)

    TRAIN_PATH = "/kaggle/input/competitions/playground-series-s6e4/train.csv"
    TEST_PATH = "/kaggle/input/competitions/playground-series-s6e4/test.csv"
    SAMPLE_PATH = "/kaggle/input/competitions/playground-series-s6e4/sample_submission.csv"

    TARGET = "Irrigation_Need"
    ID_COL = "id"

    SEED = 42
    N_SPLITS = 5

    TARGET_MAP = {"Low": 0, "Medium": 1, "High": 2}
    INV_TARGET_MAP = {0: "Low", 1: "Medium", 2: "High"}

    REF_NPY_PATH = "/kaggle/input/datasets/mizushimatoshihiko/npy-files/"
    REF_OOF = {
        "018": REF_NPY_PATH + "oof_cat_pairwise_te_bias_from_shared_min_proba_biased.npy",
        "046b": REF_NPY_PATH + "oof_xgb_digit_orderedte_min_optuna_biased_refined.npy",
        "025": REF_NPY_PATH + "oof_lgb_digit_te_min_proba_biased.npy",
        "053": REF_NPY_PATH + "oof_lgb_digit_te_threshold_min_from_family_proba_biased.npy",
        "056": REF_NPY_PATH + "oof_cat_pairwise_te_threshold_min_from_family_gpu_default_fix_proba_biased.npy",
        "057": REF_NPY_PATH + "oof_xgb046b_high_interaction_min_biased.npy",
        "060": REF_NPY_PATH + "oof_cat_depth1_formula_min_biased.npy",
    }
    
    _061_DIR = REF_NPY_PATH + "exp_20260428_061_ashish_xgb_orig_te_bias/"
    _061_OOF = REF_NPY_PATH + "oof_ashish_xgb_orig_te_raw.npy"
    _061_PRD = REF_NPY_PATH + "pred_ashish_xgb_orig_te_raw.npy"
    BEST_BIAS_061_NPY = _061_DIR + "best_class_bias.npy"

In [4]:
# ============================================================
# Utils
# ============================================================

def seed_everything(seed: int):
    random.seed(seed)
    np.random.seed(seed)


def save_json(obj, path):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)


def public_preds(proba: np.ndarray, bias: np.ndarray) -> np.ndarray:
    return np.argmax(np.log(np.clip(proba, 1e-15, 1.0)) + bias.reshape(1, -1), axis=1)


def tune_bias_coordinate_descent(
    proba: np.ndarray,
    y_true: np.ndarray,
    steps=(1.0, 0.5, 0.2, 0.1, 0.05, 0.02, 0.01, 0.005, 0.002),
):
    best_bias = np.zeros(proba.shape[1], dtype=np.float64)
    best_score = balanced_accuracy_score(y_true, public_preds(proba, best_bias))
    history = [best_score]

    for step in steps:
        improved = True
        while improved:
            improved = False
            for ci in range(proba.shape[1]):
                for direction in (-1.0, 1.0):
                    cand = best_bias.copy()
                    cand[ci] += direction * step
                    score = balanced_accuracy_score(y_true, public_preds(proba, cand))
                    if score > best_score + 1e-9:
                        best_bias = cand
                        best_score = score
                        improved = True
                        history.append(best_score)
                        print(f"bias step={step:.4f} score={best_score:.9f} bias={best_bias}")

    return best_bias, float(best_score), history


def apply_bias_to_proba(proba: np.ndarray, bias: np.ndarray) -> np.ndarray:
    logp = np.log(np.clip(proba, 1e-15, 1.0)) + bias.reshape(1, -1)
    logp = logp - logp.max(axis=1, keepdims=True)
    e = np.exp(logp)
    return (e / e.sum(axis=1, keepdims=True)).astype(np.float32)


def mean_raw_corr(a, b):
    vals = []
    for c in range(a.shape[1]):
        vals.append(np.corrcoef(a[:, c], b[:, c])[0, 1])
    return float(np.mean(vals))


def mean_rank_corr(a, b):
    vals = []
    for c in range(a.shape[1]):
        ra = pd.Series(a[:, c]).rank(method="average").to_numpy()
        rb = pd.Series(b[:, c]).rank(method="average").to_numpy()
        vals.append(np.corrcoef(ra, rb)[0, 1])
    return float(np.mean(vals))


seed_everything(CFG.SEED)

In [5]:
# ============================================================
# Load
# ============================================================
train = pd.read_csv(CFG.TRAIN_PATH)
test = pd.read_csv(CFG.TEST_PATH)
train[CFG.TARGET] = train[CFG.TARGET].map(CFG.TARGET_MAP).astype(int)
y_train = train[CFG.TARGET].astype(int)

oof_raw = np.load(CFG._061_OOF)
pred_raw = np.load(CFG._061_PRD)

In [6]:
# ============================================================
# Bias Refine - fast local grid
# ============================================================

def fast_balanced_accuracy_from_pred(y_true: np.ndarray, y_pred: np.ndarray, n_classes: int = 3) -> float:
    score = 0.0
    for c in range(n_classes):
        mask = (y_true == c)
        denom = mask.sum()
        if denom == 0:
            continue
        score += ((y_pred[mask] == c).sum() / denom) / n_classes
    return float(score)


y_np = y_train.to_numpy() if hasattr(y_train, "to_numpy") else np.asarray(y_train)

log_oof = np.log(np.clip(oof_raw, 1e-15, 1.0))

# 061のbest_biasが読めない場合に備える
if os.path.exists(CFG.BEST_BIAS_061_NPY):
    base_bias = np.load(CFG.BEST_BIAS_061_NPY).astype(np.float64)
else:
    base_bias = np.array([-1.357, -1.193, 0.0], dtype=np.float64)

# Highを基準に固定
base_bias = base_bias - base_bias[2]
base_bias[2] = 0.0

parent_pred = np.argmax(log_oof + base_bias.reshape(1, -1), axis=1)
parent_score = fast_balanced_accuracy_from_pred(y_np, parent_pred)

print("base_bias:", base_bias)
print("parent_score:", parent_score)

# stage 1: 0.001 grid
grid0 = np.arange(base_bias[0] - 0.03, base_bias[0] + 0.0301, 0.001)
grid1 = np.arange(base_bias[1] - 0.03, base_bias[1] + 0.0301, 0.001)

best_score = parent_score
best_bias = base_bias.copy()

for b0 in grid0:
    for b1 in grid1:
        bias = np.array([b0, b1, 0.0], dtype=np.float64)
        pred = np.argmax(log_oof + bias.reshape(1, -1), axis=1)
        sc = fast_balanced_accuracy_from_pred(y_np, pred)
        if sc > best_score + 1e-12:
            best_score = sc
            best_bias = bias.copy()

print("1st stage bias refine:")
print(best_score, best_bias)

# stage 2: 0.0002 grid
b0, b1, _ = best_bias

grid0 = np.arange(b0 - 0.005, b0 + 0.00501, 0.0002)
grid1 = np.arange(b1 - 0.005, b1 + 0.00501, 0.0002)

best_score2 = best_score
best_bias2 = best_bias.copy()

for x0 in grid0:
    for x1 in grid1:
        bias = np.array([x0, x1, 0.0], dtype=np.float64)
        pred = np.argmax(log_oof + bias.reshape(1, -1), axis=1)
        sc = fast_balanced_accuracy_from_pred(y_np, pred)
        if sc > best_score2 + 1e-12:
            best_score2 = sc
            best_bias2 = bias.copy()

print("2nd stage bias refine:")
print(best_score2, best_bias2)

oof_biased = apply_bias_to_proba(oof_raw, best_bias2)
pred_biased = apply_bias_to_proba(pred_raw, best_bias2)

base_bias: [-1.35699999 -1.19299996  0.        ]
parent_score: 0.9801615755373589
1st stage bias refine:
0.9801673900932395 [-1.34799999 -1.19199996  0.        ]
2nd stage bias refine:
0.9801687843616809 [-1.34739999 -1.19139996  0.        ]


In [7]:
# ============================================================
# Save artifacts
# ============================================================

np.save(CFG.SAVE_DIR / "oof_ashish_xgb_orig_te_bias_refined.npy", oof_biased)
np.save(CFG.SAVE_DIR / "pred_ashish_xgb_orig_te_bias_refined.npy", pred_biased)

np.save(CFG.SAVE_DIR / "best_class_bias_refined.npy", best_bias2.astype(np.float32))

submission = pd.DataFrame(
    {
        CFG.ID_COL: test[CFG.ID_COL],
        CFG.TARGET: pd.Series(np.argmax(pred_biased, axis=1)).map(CFG.INV_TARGET_MAP),
    }
)
submission.to_csv(CFG.SAVE_DIR / f"submission_{CFG.EXP_NAME}.csv", index=False)

print("submission distribution:")
print(submission[CFG.TARGET].value_counts(normalize=True))

raw_cv = balanced_accuracy_score(y_np, np.argmax(oof_raw, axis=1))
parent_bias_cv = parent_score
refined_cv = best_score2
biased_check = balanced_accuracy_score(y_np, np.argmax(oof_biased, axis=1))

cv_result = {
    "exp_id": CFG.EXP_ID,
    "exp_name": CFG.EXP_NAME,
    "parent": "exp_20260428_061_ashish_xgb_orig_te_bias",
    "raw_cv": float(raw_cv),
    "parent_bias_cv": float(parent_bias_cv),
    "refined_cv": float(refined_cv),
    "biased_check": float(biased_check),
    "gain_vs_raw": float(refined_cv - raw_cv),
    "gain_vs_parent_bias": float(refined_cv - parent_bias_cv),
    "base_bias": [float(x) for x in base_bias],
    "stage1_best_bias": [float(x) for x in best_bias],
    "stage2_best_bias": [float(x) for x in best_bias2],
    "search": {
        "stage1": {
            "center": [float(x) for x in base_bias],
            "width": 0.03,
            "step": 0.001,
            "fixed_class_2_bias": 0.0,
        },
        "stage2": {
            "center": [float(x) for x in best_bias],
            "width": 0.005,
            "step": 0.0002,
            "fixed_class_2_bias": 0.0,
        },
    },
    "notes": {
        "type": "postprocess_only",
        "no_retraining": True,
        "adoption_rule": "adopt only if Public LB exceeds 061 single LB 0.98109",
    },
}

save_json(cv_result, CFG.SAVE_DIR / "cv_result.json")

submission distribution:
Irrigation_Need
Low       0.590307
Medium    0.371493
High      0.038200
Name: proportion, dtype: float64


In [8]:
# ============================================================
# Correlation vs existing OOF
# ============================================================

corr_rows = []
for name, path in CFG.REF_OOF.items():
    if os.path.exists(path):
        ref = np.load(path)
        corr_rows.append(
            {
                "ref": name,
                "raw_corr_mean": mean_raw_corr(oof_biased, ref),
                "rank_corr_mean": mean_rank_corr(oof_biased, ref),
            }
        )
    else:
        print("missing ref:", name, path)

corr_df = pd.DataFrame(corr_rows)
if len(corr_df):
    corr_df = corr_df.sort_values("rank_corr_mean", ascending=False)
    corr_df.to_csv(CFG.SAVE_DIR / "corr_vs_existing_oof.csv", index=False)
    display(corr_df)

print("saved to:", CFG.SAVE_DIR)
print("Done.")

,ref,raw_corr_mean,rank_corr_mean
5,057,0.991397,0.964924
1,046b,0.991481,0.961018
3,053,0.992548,0.956728
2,025,0.991522,0.949087
0,018,0.989579,0.946778
4,056,0.992102,0.943069
6,060,0.972226,0.927957


saved to: /kaggle/working/exp_20260429_061b_ashish_xgb_orig_te_bias_refined
Done.
